In [ ]:
import os
import pandas as pd
import cdsapi 
import xarray as xr

os.environ['CDSAPI_RC'] = r"D:\\Shukra_sir\\CDSAPI_ERA5\\.cdsapirc.txt"

output_folder_Chlorophyll = r"D:\\Shukra_sir\\ERA5 MeteoData\\Chlorophyll_12H"  
output_folder_DOC = r"D:\\Shukra_sir\\ERA5 MeteoData\\DOC_12H"
output_folder_TSS = r"D:\\Shukra_sir\\ERA5 MeteoData\\TSS_12H"

matchup_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_Chlorophyll"
matchup_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_DOC"
matchup_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_Pheophytin"
matchup_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_TSS"

ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
folder = matchup_Chlorophyll

def get_data(folder):
    param = ALL_PARAMETERS[0]
    file_name = f"NewCombinedFiles12H_{param}.csv"
    file_name_path = os.path.join(folder, file_name)
    combined_df = pd.read_csv(file_name_path)

    lat_min, lat_max = combined_df['latitude'].min(), combined_df['latitude'].max()
    lon_min, lon_max = combined_df['longitude'].min(), combined_df['longitude'].max()
    year_min, year_max = combined_df['year'].min(), combined_df['year'].max()

    unique_years = combined_df['year'].unique().tolist()
    unique_years.sort()
    unique_months = combined_df['month'].unique().tolist()
    unique_months.sort()
    unique_days = combined_df['day'].unique().tolist()
    unique_days.sort()
    unique_hours = combined_df['hour'].unique().tolist()
    unique_hours.sort()

    # Add buffer (1 degrees)
    area = [lat_max + 1, lon_min - 1, lat_min - 1, lon_max + 1]  # N, W, S, E
    return area, year_min, year_max, unique_years, unique_months, unique_days, unique_hours, param


area = get_data(folder)[0]
year_start = get_data(folder)[1]
year_end = get_data(folder)[2]
#unique_years = [2023]
unique_years = get_data(folder)[3]
unique_months = get_data(folder)[4]
unique_days = get_data(folder)[5]
unique_hours = get_data(folder)[6]

print (area)
print (year_start)
print (year_end)
print (unique_years)
print (unique_months)
print(unique_days) 
print(unique_hours)
print(get_data(folder)[7])

def get_GRIB(folder):

    param = ALL_PARAMETERS[3]
    area = get_data(folder)[0]
    unique_years = get_data(folder)[3]
    unique_months = get_data(folder)[4]
    unique_days = get_data(folder)[5]
    unique_hours = get_data(folder)[6]
    year = 2024 #change year manually from 2017 to 2024. Loop is avoided to prevent server overload (specifically, to avoid RuntimeError: Mars runtime error)
    output_folder = output_folder_DOC

    c = cdsapi.Client()

    print(f"Downloading ERA5 data_{year} for {param}...")
    grib_file = os.path.join(output_folder, f'ERA5GRIB_{year}_{param}.grib')
        
    c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": ['2m_temperature', '2m_dewpoint_temperature', '10m_v_component_of_neutral_wind', '10m_u_component_of_neutral_wind'],
                "year": f"{year}",
                "month": [f"{m:02d}" for m in range(1,13)],
                "day": [f"{d:02d}" for d in range(1,32)],
                "time": [f"{h:02d}:00" for h in unique_hours],   # ONLY unique hours needed
                "format": "grib",
                "area": area
            },
            grib_file)
        
    print(f"GRIB saved for {year} and for {param} as: {grib_file}")
    


#get_GRIB(matchup_Chlorophyll)
#get_GRIB(matchup_TSS)
#get_GRIB(matchup_DOC)
#get_GRIB(matchup_Pheophytin)

[np.float64(39.81221), np.float64(-123.0811), np.float64(36.06645), np.float64(-120.07219)]
2017
2024
[2017, 2018, 2019, 2020, 2021, 2022, 2023, 2024]
[1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12]
[1, 2, 3, 4, 6, 7, 8, 9, 10, 11, 12, 13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24, 25, 26, 27, 28, 29, 30, 31]
[18, 19]
Chlorophyll


In [89]:
import xarray as xr

ds1 = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\era5_test2.nc")
ds = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\TSS_12H\\ERA5GRIB_2024_TSS.grib")

print(ds)



<xarray.Dataset> Size: 16MB
Dimensions:     (time: 732, latitude: 38, longitude: 36)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 6kB 2024-01-01T18:00:00 ... 2024-12-31T...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 304B 42.59 42.34 42.09 ... 33.84 33.59 33.34
  * longitude   (longitude) float64 288B -125.1 -124.8 -124.6 ... -116.6 -116.3
    valid_time  (time) datetime64[ns] 6kB ...
Data variables:
    t2m         (time, latitude, longitude) float32 4MB ...
    d2m         (time, latitude, longitude) float32 4MB ...
    v10n        (time, latitude, longitude) float32 4MB ...
    u10n        (time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range Weather Forecasts
    GRIB_subCentre:          0
    Conventions:             CF-1.7
    institution:             Europea

d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


In [ ]:
import xarray as xr
import pandas as pd
import scipy 

#ds1 = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\era5_test2.nc")
ds = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final\\ERA5GRIB_2017_Chlorophyll.grib")

#print(ds1)
#print(ds.variables)

lat = 36.0142
lon = -119.9769
timestamps = pd.Timestamp("2019-11-23 13:00:00")
result = timestamps - pd.Timedelta("12h")


point1 = ds["t2m"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic"
    )

point2 = ds["d2m"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic"
    )

point3 = ds["v10n"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic"
    )

point4 = ds["u10n"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic"
    )

    
temperature = float(point1.interp(
    time=timestamps,
    method = "cubic"
).values)

dewpoint = float(point2.interp(
    time=timestamps,
    method = "cubic"
).values)

v10 = float(point3.interp(
    time=timestamps,
    method = "cubic"
).values)

u10 = float(point4.interp(
    time=timestamps,
    method = "cubic"
).values)




"""
point1 = ds["t2m"].interp(
    latitude=lat,
    longitude=lon,
    time=timestamps,
    method="cubic")
    
point2 = ds["d2m"].interp(
    latitude=lat,
    longitude=lon,
    time=timestamps,
    method="cubic")
    
point3 = ds["v10n"].interp(
    latitude=lat,
    longitude=lon,
    time=timestamps,
    method="cubic")
    
point4 = ds["u10n"].interp(
    latitude=lat,
    longitude=lon,
    time=timestamps,
    method="cubic")

temperature = float(point1.values)
dewpoint = float(point2.values)
v10 = float(point3.values)
u10 = float(point4.values)
"""

print("Temp:", temperature, "K")
print("Dew Point Temp:", dewpoint, "K")
print("V10:", v10, "m/s")
print("U10:", u10, "m/s")

print(ds)

#print(ds.time.values[:5])
#print(ds.valid_time.values[:5])

#print(ds.latitude.values)
#print(ds.longitude.values)

""" 
precipitation2 = float(point.sel(
    time=pd.Timestamp("2017-06-02 06:00:00"),
    step=pd.Timedelta("12h"),
    method = "nearest"
).values)
"""






Ignoring index file 'D:\\Shukra_sir\\ERA5 MeteoData\\TSS_12H_Final\\ERA5GRIB_2019_TSS.grib.5b7b6.idx' incompatible with GRIB file
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


Temp: 284.26220179054087 K
Dew Point Temp: 275.3755970241193 K
V10: -1.3886944777127805 m/s
U10: 0.3334320478576013 m/s
<xarray.Dataset> Size: 16MB
Dimensions:     (time: 730, latitude: 38, longitude: 36)
Coordinates:
    number      int64 8B ...
  * time        (time) datetime64[ns] 6kB 2019-01-01T18:00:00 ... 2019-12-31T...
    step        timedelta64[ns] 8B ...
    surface     float64 8B ...
  * latitude    (latitude) float64 304B 42.59 42.34 42.09 ... 33.84 33.59 33.34
  * longitude   (longitude) float64 288B -125.1 -124.8 -124.6 ... -116.6 -116.3
    valid_time  (time) datetime64[ns] 6kB ...
Data variables:
    t2m         (time, latitude, longitude) float32 4MB ...
    d2m         (time, latitude, longitude) float32 4MB ...
    v10n        (time, latitude, longitude) float32 4MB ...
    u10n        (time, latitude, longitude) float32 4MB ...
Attributes:
    GRIB_edition:            1
    GRIB_centre:             ecmf
    GRIB_centreDescription:  European Centre for Medium-Range W

' \nprecipitation2 = float(point.sel(\n    time=pd.Timestamp("2017-06-02 06:00:00"),\n    step=pd.Timedelta("12h"),\n    method = "nearest"\n).values)\n'

In [ ]:
import xarray as xr
import pandas as pd
import scipy 

ds = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final\\ERA5GRIB_2019_Chlorophyll.grib")


lat = 38.042889
lon = -121.920111
timestamps = pd.Timestamp("2019-01-10 18:00:00")
result = timestamps - pd.Timedelta("12h")


# --- Flatten time dimension ----
tp_flat = ds["tp"].stack(datetime=("time", "step"))
tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())

# --- Interpolate spatially ---
point = tp_flat.interp(
    latitude=lat,
    longitude=lon,
    method="linear"
)

# --- Interpolate in time ---
precipitation = point.interp(
    datetime=timestamps,
    method="linear"
).item()

#print("Precipitation:", precipitation, "m")


nearest = tp_flat.sel(
    latitude=lat,
    longitude=lon,
    method="nearest"
)
#print(nearest)

#print(ds.latitude.min().values, ds.latitude.max().values)
#print(ds.longitude.min().values, ds.longitude.max().values)

"""
point = ds["tp"].interp(
        latitude=lat,
        longitude=lon,
        method="linear"
    )
"""

"""
precipitation = float(point.interp(
    time=timestamps,
    method = "linear"
).values)
"""

#print("Precipitation:", precipitation, "m")

#print(ds)

print(ds["tp"].sel(
    latitude=lat,
    longitude=lon,
    method="nearest"
).values)




Ignoring index file 'D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final\\ERA5GRIB_2019_Chlorophyll.grib.5b7b6.idx' incompatible with GRIB file
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


[[nan  0.]
 [ 0. nan]
 [nan  0.]
 ...
 [ 0. nan]
 [nan  0.]
 [ 0. nan]]


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\2269830539.py:16: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())


In [81]:
ds = xr.open_dataset("D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final\\ERA5GRIB_2019_Chlorophyll.grib")


lat = 38.32885
lon = -121.66753
timestamps = pd.Timestamp("2019-07-09 19:00:00")



tp_flat = ds["tp"].stack(datetime=("time", "step"))
tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())

# Select the nearest grid point (spatial) and nearest time
point = tp_flat.sel(
    latitude=lat,
    longitude=lon,
    method="nearest"
)
precipitation = float(point.sel(datetime=timestamps, method="nearest").values)

# Print results
print("Precipitation (nearest point/time):", precipitation, "m")
print("Precipitation in mm:", precipitation * 1000, "mm")

Ignoring index file 'D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final\\ERA5GRIB_2019_Chlorophyll.grib.5b7b6.idx' incompatible with GRIB file
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timedelta. To silence this warning, set decode_timedelta to True, False, or a 'CFTimedeltaCoder' instance.
  vars, attrs, coord_names = xr.conventions.decode_cf_variables(


Precipitation (nearest point/time): 6.839632987976074e-06 m
Precipitation in mm: 0.006839632987976074 mm


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\3345314430.py:11: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())


In [82]:
import numpy as np

tp_flat = ds["tp"].stack(datetime=("time", "step"))
tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())

# Select nearest grid point
point = tp_flat.sel(latitude=lat, longitude=lon, method="nearest")

# Convert to numpy array
tp_values = point.values

# Find indices where precipitation > 0
nonzero_indices = np.where(tp_values > 0)[0]

# Check if there are any non-zero values
if len(nonzero_indices) > 0:
    print(f"Found {len(nonzero_indices)} non-zero precipitation values.")
    # Optional: show first few
    for idx in nonzero_indices[:200]:
        print(f"{point.datetime[idx].values}: {tp_values[idx]*1000:.2f} mm")
else:
    print("All precipitation values are zero at this location.")

Found 203 non-zero precipitation values.
2019-01-05T18:00:00.000000000: 0.95 mm
2019-01-05T19:00:00.000000000: 1.03 mm
2019-01-06T18:00:00.000000000: 0.35 mm
2019-01-06T19:00:00.000000000: 1.35 mm
2019-01-07T18:00:00.000000000: 0.01 mm
2019-01-07T19:00:00.000000000: 0.00 mm
2019-01-08T18:00:00.000000000: 0.00 mm
2019-01-08T19:00:00.000000000: 0.32 mm
2019-01-09T18:00:00.000000000: 0.29 mm
2019-01-09T19:00:00.000000000: 0.40 mm
2019-01-10T18:00:00.000000000: 0.00 mm
2019-01-10T19:00:00.000000000: 0.00 mm
2019-01-11T18:00:00.000000000: 0.00 mm
2019-01-11T19:00:00.000000000: 0.00 mm
2019-01-14T18:00:00.000000000: 0.01 mm
2019-01-15T18:00:00.000000000: 0.93 mm
2019-01-15T19:00:00.000000000: 1.48 mm
2019-01-16T18:00:00.000000000: 0.07 mm
2019-01-16T19:00:00.000000000: 0.24 mm
2019-01-17T18:00:00.000000000: 0.23 mm
2019-01-17T19:00:00.000000000: 0.46 mm
2019-01-19T19:00:00.000000000: 0.00 mm
2019-01-20T18:00:00.000000000: 0.68 mm
2019-01-20T19:00:00.000000000: 1.04 mm
2019-01-28T19:00:00.000

C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\939541423.py:4: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())


In [ ]:
import pandas as pd
import xarray as xr
import os
import scipy

GRIB_folder_Chlorophyll = r"D:\\Shukra_sir\\ERA5 MeteoData\\Chlorophyll_12H_Final"  
GRIB_folder_DOC = r"D:\\Shukra_sir\\ERA5 MeteoData\\DOC_12H_Final"
GRIB_folder_TSS = r"D:\\Shukra_sir\\ERA5 MeteoData\\TSS_12H_Final"
GRIB_folder_Pheophytin = r"D:\\Shukra_sir\\ERA5 MeteoData\\TSS_12H_Final"

matchup_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_Chlorophyll"
matchup_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_DOC"
matchup_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_Pheophytin"
matchup_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_TSS"

output_folder_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Chlorophyll"
output_folder_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_DOC"
output_folder_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_TSS"
output_folder_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Pheophytin"

ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
VARIABLES = ['Temperature', 'DewPoint Temperature', 'V10', 'U10']
VARIABLES_CODE = ['t2m', 'd2m', 'v10n', 'u10n']

class GRIB_Matcher:
    def __init__(self, parameter):
        self.parameter = parameter
    
    def get_value_from_grib(self, ds, lat, lon, timestamp):
        
        point1 = ds["t2m"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic")
        
        point2 = ds["d2m"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic")
        
        point3 = ds["v10n"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic")
        
        point4 = ds["u10n"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic")

        temperature = float(point1.interp(
            time=timestamp,
            method = "cubic"
        ).values)
        
        dewpoint = float(point2.interp(
            time=timestamp,
            method = "cubic"
        ).values)
        
        v10 = float(point3.interp(
            time=timestamp,
            method = "cubic"
        ).values)
        
        u10 = float(point4.interp(
            time=timestamp,
            method = "cubic"
        ).values)

        return temperature, dewpoint, v10, u10


    def match_GRIB(self):
        
        PARAM = self.parameter
        print(f"Matching ERA5 GRIB data for parameter: {PARAM}...")

        if PARAM == 'Chlorophyll':
            combined_file_name = f"NewCombinedFiles12H_Chlorophyll.csv"
            combined_file_name_path = os.path.join(matchup_Chlorophyll, combined_file_name)
            grib_folder = GRIB_folder_Chlorophyll
            output_folder = output_folder_Chlorophyll

        elif PARAM == 'TSS':
            combined_file_name = f"NewCombinedFiles12H_TSS.csv"
            combined_file_name_path = os.path.join(matchup_TSS, combined_file_name)
            grib_folder = GRIB_folder_TSS
            output_folder = output_folder_TSS

        elif PARAM == 'DOC':    
            combined_file_name = f"NewCombinedFiles12H_DOC.csv"
            combined_file_name_path = os.path.join(matchup_DOC, combined_file_name)
            grib_folder = GRIB_folder_DOC
            output_folder = output_folder_DOC

        elif PARAM == 'Pheophytin':
            combined_file_name = f"NewCombinedFiles12H_Pheophytin.csv"
            combined_file_name_path = os.path.join(matchup_Pheophytin, combined_file_name)
            grib_folder = GRIB_folder_Pheophytin
            output_folder = output_folder_Pheophytin
        
        df = pd.read_csv(combined_file_name_path)
        
        Temperature = []
        Dewpoint = []
        v10_list = []
        u10_list = []
        count = 0
        for __, row in df.iterrows():
            lat = row['latitude']
            lon = row['longitude']
            year = row['year']
            month = row['month']
            day = row['day']
            hour = row['hour']
            minute = row['minute']
            second = row['second']

            time = f"{year}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:02d}"
            timestamp = pd.Timestamp(time)

            grib_file_path = os.path.join(grib_folder, f'ERA5GRIB_{year}_{PARAM}.grib')
            ds = xr.open_dataset(grib_file_path)

            temperature, dewpoint, v10_val, u10_val = self.get_value_from_grib(ds, lat, lon, timestamp)
            ds.close()


            #temperature = self.get_value_from_grib(ds, lat, lon, timestamp)[0]
            #dewpoint = self.get_value_from_grib(ds, lat, lon, timestamp)[1]
            #v10_val = self.get_value_from_grib(ds, lat, lon, timestamp)[2]
            #u10_val = self.get_value_from_grib(ds, lat, lon, timestamp)[3]

            Temperature.append(temperature)
            Dewpoint.append(dewpoint)
            v10_list.append(v10_val)
            u10_list.append(u10_val)
            count +=1
            print("Processed for row:", count)

        
        df["Temperature"] = Temperature
        df["DewPoint"] = Dewpoint
        df["v10n"] = v10_list
        df["u10n"] = u10_list

        output_file_name = f"ERA5_Sentinel_{PARAM}_12H.csv"
        output_file_name_path = os.path.join(output_folder, output_file_name)
        df.to_csv(output_file_name_path, index=False)
        
        print(f"Matched data saved as: {output_file_name_path}")


#For Chlorophyll
#ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
#param = ALL_PARAMETERS[0]
#GRIB_Matcher(param).match_GRIB()

#FOR TSS
#ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
#param = ALL_PARAMETERS[1]
#GRIB_Matcher(param).match_GRIB()

#FOR DOC
#ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
#param = ALL_PARAMETERS[2]
#GRIB_Matcher(param).match_GRIB()




In [ ]:
chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Chlorophyll\\ERA5_Sentinel_Chlorophyll_12H.csv"
pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Pheophytin\\ERA5_Sentinel_Pheophytin_12H.csv"


chlorophyll_combined = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_Chlorophyll\\NewCombinedFiles12H_Chlorophyll.csv"
pheophytin_combined = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_Pheophytin\\NewCombinedFiles12H_Pheophytin.csv"

df1 = pd.read_csv(chlorophyll_combined)
df2 = pd.read_csv(pheophytin_combined)

df = pd.read_csv (chlorophyll)


Temperature = []
Dewpoint = []
v10_list = []
u10_list = []
count = 0

for __, row in df.iterrows():
    temp = row['Temperature']
    dew = row['DewPoint']
    v10 = row['v10n']
    u10 = row['u10n']

    Temperature.append(temp)
    Dewpoint.append(dew)
    v10_list.append(v10)
    u10_list.append(u10)
    count +=1

    print("Processed for row:", count)

df2["Temperature"] = Temperature
df2["DewPoint"] = Dewpoint
df2["v10n"] = v10_list
df2["u10n"] = u10_list

df2.to_csv(pheophytin, index=False)
print(f"Done")






Processed for row: 1
Processed for row: 2
Processed for row: 3
Processed for row: 4
Processed for row: 5
Processed for row: 6
Processed for row: 7
Processed for row: 8
Processed for row: 9
Processed for row: 10
Processed for row: 11
Processed for row: 12
Processed for row: 13
Processed for row: 14
Processed for row: 15
Processed for row: 16
Processed for row: 17
Processed for row: 18
Processed for row: 19
Processed for row: 20
Processed for row: 21
Processed for row: 22
Processed for row: 23
Processed for row: 24
Processed for row: 25
Processed for row: 26
Processed for row: 27
Processed for row: 28
Processed for row: 29
Processed for row: 30
Processed for row: 31
Processed for row: 32
Processed for row: 33
Processed for row: 34
Processed for row: 35
Processed for row: 36
Processed for row: 37
Processed for row: 38
Processed for row: 39
Processed for row: 40
Processed for row: 41
Processed for row: 42
Processed for row: 43
Processed for row: 44
Processed for row: 45
Processed for row: 

In [34]:
#To download GRIB with only "Precipitation" variable. Then, add another column of "Precipitation" on the final excel files.

import os
import cdsapi
import pandas as pd

os.environ['CDSAPI_RC'] = r"D:\\Shukra_sir\\CDSAPI_ERA5\\.cdsapirc.txt"

output_folder_Chlorophyll = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final" 
output_folder_DOC = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\DOC_12H_Final"
output_folder_TSS = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\TSS_12H_Final"
output_folder_Pheophytin = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Pheophytin_12H_Final"


matchup_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_Chlorophyll"
matchup_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_DOC"
matchup_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_Pheophytin"
matchup_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_TSS"

ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']

def get_data(folder, param):
    #param = ALL_PARAMETERS[0]
    file_name = f"NewCombinedFiles12H_{param}.csv"
    file_name_path = os.path.join(folder, file_name)
    combined_df = pd.read_csv(file_name_path)

    lat_min, lat_max = combined_df['latitude'].min(), combined_df['latitude'].max()
    lon_min, lon_max = combined_df['longitude'].min(), combined_df['longitude'].max()
    year_min, year_max = combined_df['year'].min(), combined_df['year'].max()

    unique_years = combined_df['year'].unique().tolist()
    unique_years.sort()
    unique_months = combined_df['month'].unique().tolist()
    unique_months.sort()
    unique_days = combined_df['day'].unique().tolist()
    unique_days.sort()
    unique_hours = combined_df['hour'].unique().tolist()
    unique_hours.sort()

    # Add buffer (1 degrees)
    area = [lat_max + 1, lon_min - 1, lat_min - 1, lon_max + 1]  # N, W, S, E
    return area, year_min, year_max, unique_years, unique_months, unique_days, unique_hours, param


def get_GRIB(folder, parameter):

    #param = ALL_PARAMETERS[3]
    param = parameter

    if param == 'Chlorophyll':
        output_folder = output_folder_Chlorophyll
    elif param == 'TSS':
        output_folder = output_folder_TSS     
    elif param == 'DOC':
        output_folder = output_folder_DOC
    elif param == 'Pheophytin':
        output_folder = output_folder_Pheophytin


    area = get_data(folder, param)[0]
    unique_years = get_data(folder, param)[3]
    unique_months = get_data(folder, param)[4]
    unique_days = get_data(folder, param)[5]
    unique_hours = get_data(folder, param)[6]
    year = 2024 #change year manually from 2017 to 2024. Loop is avoided to prevent server overload (specifically, to avoid RuntimeError: Mars runtime error)
    #output_folder = output_folder_DOC

    c = cdsapi.Client()

    print(f"Downloading ERA5 data_{year} for {param}...")
    grib_file = os.path.join(output_folder, f'ERA5GRIB_{year}_{param}.grib')
        
    c.retrieve(
            "reanalysis-era5-single-levels",
            {
                "product_type": "reanalysis",
                "variable": ['total_precipitation'],
                "year": f"{year}",
                "month": [f"{m:02d}" for m in range(1,13)],
                "day": [f"{d:02d}" for d in range(1,32)],
                "time": [f"{h:02d}:00" for h in unique_hours],   # ONLY unique hours needed
                "format": "grib",
                "area": area
            },
            grib_file)
        
    print(f"GRIB saved for {year} and for {param} as: {grib_file}")
    
    
#Param = ALL_PARAMETERS[0]
#get_GRIB(matchup_Chlorophyll, Param)


#Param = ALL_PARAMETERS[1]
#get_GRIB(matchup_TSS, Param)


#Param = ALL_PARAMETERS[2]
#get_GRIB(matchup_DOC, Param)


Param = ALL_PARAMETERS[3]
get_GRIB(matchup_Pheophytin, Param)


2025-12-12 16:34:34,916 INFO [2025-12-03T00:00:00Z] To improve our C3S service, we need to hear from you! Please complete this very short [survey](https://confluence.ecmwf.int/x/E7uBEQ/). Thank you.


2025-12-12 16:34:35,958 INFO [2025-12-11T00:00:00] Please note that a dedicated catalogue entry for this dataset, post-processed and stored in Analysis Ready Cloud Optimized (ARCO) format (Zarr), is available for optimised time-series retrievals (i.e. for retrieving data from selected variables for a single point over an extended period of time in an efficient way). You can discover it [here](https://cds.climate.copernicus.eu/datasets/reanalysis-era5-single-levels-timeseries?tab=overview)
2025-12-12 16:34:35,960 INFO Request ID is d73fb94c-933d-48a4-a631-d1a2f64466c5
2025-12-12 16:34:36,277 INFO status has been updated to accepted
2025-12-12 16:34:52,963 INFO status has been updated to successful
                                                                                        

GRIB saved for 2024 and for Pheophytin as: D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Pheophytin_12H_Final\ERA5GRIB_2024_Pheophytin.grib


In [ ]:
import pandas as pd
import os 
import xarray as xr

precip_grib_folder_Chlorophyll = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Chlorophyll_12H_Final" 
precip_grib_folder_DOC = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\DOC_12H_Final"
precip_grib_output_folder_TSS = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\TSS_12H_Final"
precip_grib_output_folder_Pheophytin = r"D:\\Shukra_sir\\ERA5 MeteoData\\Grib for Precipitation\\Pheophytin_12H_Final"

matchup_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_Chlorophyll"
matchup_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_DOC"
matchup_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Matchfiles12H_Pheophytin"
matchup_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\MatchFiles12H_TSS"

#output_folder_Chlorophyll = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Chlorophyll"
#output_folder_DOC = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_DOC"
#output_folder_TSS = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_TSS"
#output_folder_Pheophytin = "D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\Meteorology_Pheophytin"

output = r"D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\With_Precipitation"


ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
VARIABLES = ['Temperature', 'DewPoint Temperature', 'V10', 'U10']
VARIABLES_CODE = ['t2m', 'd2m', 'v10n', 'u10n']
PRECIPITATION = ['tp']

class GRIB_Matcher:
    def __init__(self, parameter):
        self.parameter = parameter
    
    def get_value_from_grib(self, ds, lat, lon, timestamp):
        
        """
        point = ds["tp"].interp(
        latitude=lat,
        longitude=lon,
        method="cubic")
        """

        tp_flat = ds["tp"].stack(datetime=("time", "step"))
        tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
        
        # Select the nearest grid point (spatial) and nearest time
        point = tp_flat.sel(
             latitude=lat,
             longitude=lon,
             method="nearest"
             )
        
        precipitation = float(point.sel(datetime=timestamp, method="nearest").values)

        """
        precipitation = float(point.interp(
            time=timestamp,
            method = "cubic"
        ).values)
        """
        return precipitation


    def match_GRIB(self):
        
        PARAM = self.parameter
        print(f"Matching ERA5 GRIB data for parameter: {PARAM}...")

        if PARAM == 'Chlorophyll':
            combined_file_name = f"NewCombinedFiles12H_Chlorophyll.csv"
            combined_file_name_path = os.path.join(matchup_Chlorophyll, combined_file_name)
            grib_folder = precip_grib_folder_Chlorophyll
            output_folder = output

        elif PARAM == 'TSS':
            combined_file_name = f"NewCombinedFiles12H_TSS.csv"
            combined_file_name_path = os.path.join(matchup_TSS, combined_file_name)
            grib_folder = precip_grib_output_folder_TSS
            output_folder = output

        elif PARAM == 'DOC':    
            combined_file_name = f"NewCombinedFiles12H_DOC.csv"
            combined_file_name_path = os.path.join(matchup_DOC, combined_file_name)
            grib_folder = precip_grib_folder_DOC
            output_folder = output

        elif PARAM == 'Pheophytin':
            combined_file_name = f"NewCombinedFiles12H_Pheophytin.csv"
            combined_file_name_path = os.path.join(matchup_Pheophytin, combined_file_name)
            grib_folder = precip_grib_output_folder_Pheophytin
            output_folder = output
        
        df = pd.read_csv(combined_file_name_path)
    
        Precipitation = []


        count = 0
        for __, row in df.iterrows():
            lat = row['latitude']
            lon = row['longitude']
            year = row['year']
            month = row['month']
            day = row['day']
            hour = row['hour']
            minute = row['minute']
            second = row['second']

            time = f"{year}-{month:02d}-{day:02d} {hour:02d}:{minute:02d}:{second:02d}"
            timestamp = pd.Timestamp(time)

            grib_file_path = os.path.join(grib_folder, f'ERA5GRIB_{year}_{PARAM}.grib')
            ds = xr.open_dataset(grib_file_path)

            precipitation = self.get_value_from_grib(ds, lat, lon, timestamp)
            ds.close()


            #temperature = self.get_value_from_grib(ds, lat, lon, timestamp)[0]
            #dewpoint = self.get_value_from_grib(ds, lat, lon, timestamp)[1]
            #v10_val = self.get_value_from_grib(ds, lat, lon, timestamp)[2]
            #u10_val = self.get_value_from_grib(ds, lat, lon, timestamp)[3]

            
            Precipitation.append(precipitation)
            count +=1
            print("Processed for row:", count)
        
        df["Precipitation (m)"] = Precipitation

        output_file_name = f"WithPrecip_ERA5_Sentinel_{PARAM}_12H.csv"
        output_file_name_path = os.path.join(output_folder, output_file_name)
        df.to_csv(output_file_name_path, index=False)
        
        print(f"Matched data saved as: {output_file_name_path}")


#For Chlorophyll
#ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
#param = ALL_PARAMETERS[0]
#GRIB_Matcher(param).match_GRIB()

#FOR TSS
ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
param = ALL_PARAMETERS[1]
GRIB_Matcher(param).match_GRIB()

#FOR DOC
#ALL_PARAMETERS = ['Chlorophyll', 'TSS', 'DOC', 'Pheophytin']
#param = ALL_PARAMETERS[2]
#GRIB_Matcher(param).match_GRIB()




C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timede

Processed for row: 559


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timede

Processed for row: 560


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timede

...

C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timede

Processed for row: 565


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
d:\Shukra_sir\WQI_Research\WQI_venv\Lib\site-packages\cfgrib\xarray_plugin.py:131: FutureWarning: In a future version, xarray will not decode timedelta values based on the presence of a timedelta-like units attribute by default. Instead it will rely on the presence of a timedelta64 dtype attribute, which is now xarray's default way of encoding timedelta64 values. To continue decoding timedeltas based on the presence of a timedelta-like units attribute, users will need to explicitly opt-in by passing True or CFTimedeltaCoder(decode_via_units=True) to decode_timede

Processed for row: 566
Processed for row: 567
Matched data saved as: D:\\Shukra_sir\\WQI_Research\\Scripts and Data\\Datasets\\Sentinel_New\\SurfaceReflectanceNew\\With_Precipitation\WithPrecip_ERA5_Sentinel_Chlorophyll_12H.csv


C:\Users\ACER\AppData\Local\Temp\ipykernel_49120\1712637356.py:42: FutureWarning: updating coordinate 'datetime' with a PandasMultiIndex would leave the multi-index level coordinates ['time', 'step'] in an inconsistent state. This will raise an error in the future. Use `.drop_vars(['datetime', 'time', 'step'])` before assigning new coordinate values.
  tp_flat = tp_flat.assign_coords(datetime=ds["valid_time"].values.flatten())
